# CS224N L01: One-hot 向量 vs 密集词向量（Dense Vector）

**Waypoint**: WP01 — 为什么需要词向量？

**官方锚点**: Slides p19-22; Notes §2.2 (one-hot vectors, Eq.1-2)

---

这个 notebook 演示一个核心问题：**为什么 one-hot 编码无法表示词义相似性？**

Slides p19-22 展示了 one-hot 和 dense vector 的概念，但只有亲手计算点积、看到具体数字，才能真正理解为什么 one-hot 的 `hotel·motel = book·fish = 0` —— 这个「所有词等距」的问题有多致命。

## 这段代码在看什么

对应 **WP01 — 为什么需要词向量？**

核心问题：one-hot 编码让每个词成为一个正交基向量（只有自己位置为 1，其余全 0），任意两个不同词的点积都是 0。这意味着模型无法区分「近义词」和「完全无关的词」。

我们会：
1. 构建 6 个词的 one-hot 向量
2. 计算它们的点积 → 全部是 0
3. 构建 toy dense 向量（模拟训练后的结果）
4. 计算余弦相似度 → 近义词接近 1，无关词接近 0 或为负

## Step 1: 导入库 & 定义词汇表

我们选 6 个词，分成 3 组：
- **hotel / motel** — 近义词（都是「汽车旅馆」）
- **cat / dog** — 同类（都是动物）
- **book / fish** — 语义无关

In [ ]:
import math

# 词汇表：6 个词
vocab = ['hotel', 'motel', 'book', 'cat', 'dog', 'fish']
V = len(vocab)  # 词汇表大小 = 6
word_to_idx = {w: i for i, w in enumerate(vocab)}  # 词 → 索引映射

print(f'词汇表 (V={V}): {vocab}')

## Step 2: 构建 One-hot 向量

对应 Slides p19：每个词是一个 V 维向量，只有自己位置为 1，其余为 0。

例如 motel 在 15000 维词汇表中 = `[0 0 0 ... 0 1 0 ... 0]`，只有第 11 个位置是 1。

这里我们用 6 维演示：

In [ ]:
# 构建 one-hot 向量：每个词只有自己的位置是 1.0，其余是 0.0
one_hot = {}
for i, w in enumerate(vocab):
    v = [0.0] * V
    v[i] = 1.0
    one_hot[w] = v
    print(f'  {w:>6s} = {v}')

## Step 3: 计算 One-hot 点积 — 全部是 0

对应 Slides p20: *"There is no natural notion of similarity for one-hot vectors!"*

对应 Notes §2.2 Eq.1-2: $v_{tea}^T v_{coffee} = v_{tea}^T v_{the} = 0$，"all words are equally dissimilar"

点积（dot product）= 对应位置相乘再求和。两个不同 one-hot 向量没有重叠的 1，所以点积永远是 0。

In [ ]:
# 计算点积：对应位置相乘再求和
def dot_product(v1, v2):
    return sum(a * b for a, b in zip(v1, v2))

# 三组词对：近义词、同类、无关
pairs = [
    ('hotel', 'motel', 'synonyms — should be similar'),
    ('cat', 'dog', 'both animals — should be somewhat similar'),
    ('book', 'fish', 'unrelated — should be dissimilar'),
]

print('--- One-hot Dot Products ---')
for w1, w2, relation in pairs:
    d = dot_product(one_hot[w1], one_hot[w2])
    print(f'  {w1:>6s} · {w2:<6s} = {d:.1f}   ({relation})')

print()
print('⚠️  ALL = 0.0 — 不同词完全正交，无法区分近义词和无关词！')

### 运行后先看哪里

看上面的输出：
- hotel·motel = 0.0（近义词应该是相似的！）
- cat·dog = 0.0（同类动物也应该是相似的！）
- book·fish = 0.0（无关词确实是 0，但近义词也是 0……）

**关键问题**：one-hot 无法区分「应该相似」和「应该不相似」的词对——它们全是 0。

## Step 4: 构建 Dense Vectors（toy 2D）

对应 Slides p22: *"We will build a dense vector for each word...measuring similarity as the vector dot product"*

这里用 2 维 toy 向量模拟训练后的结果。实际中这些值由 Word2Vec 等模型从语料中学习得到。

维度设计直觉：
- dim 0 ≈ 「人造物 vs 自然」轴
- dim 1 ≈ 「活体 vs 非活体」轴

In [ ]:
# Dense 向量（toy 2D，模拟训练后的词向量）
# 注意：实际中这些值由模型学习得到，不是手动设计的
dense = {
    'hotel': [0.80, 0.70],   # 人造物，非活体
    'motel': [0.70, 0.75],   # 和 hotel 非常接近（近义）
    'book':  [-0.50, 0.30],  # 人造物，但方向不同
    'cat':   [0.10, -0.80],  # 自然，活体
    'dog':   [0.30, -0.60],  # 和 cat 接近（同类）
    'fish':  [-0.20, -0.90], # 自然，活体，但和 cat/dog 有距离
}

print('--- Dense Vectors (toy 2D) ---')
for w, v in dense.items():
    print(f'  {w:>6s} = [{v[0]:+.2f}, {v[1]:+.2f}]')

## Step 5: 计算余弦相似度

**余弦相似度**（cosine similarity）= 两个向量方向的接近程度：

$$\cos(\theta) = \frac{v_1 \cdot v_2}{\|v_1\| \times \|v_2\|}$$

- 接近 1 = 方向几乎相同（语义很近）
- 接近 0 = 方向垂直（无关）
- 接近 -1 = 方向相反（反义）

In [ ]:
# 余弦相似度：点积 / (两个向量的长度之积)
def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(a * a for a in v2))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

print('--- Dense Cosine Similarities ---')
for w1, w2, relation in pairs:
    cs = cosine_similarity(dense[w1], dense[w2])
    print(f'  cos({w1}, {w2}) = {cs:.4f}   ({relation})')

print()
print('✅  Dense vectors successfully encode semantic similarity!')

### 输出怎么解释

| 词对 | 关系 | One-hot 点积 | Dense cos_sim |
|------|------|-------------|---------------|
| hotel-motel | 近义词 | 0.0 | 0.9949 |
| cat-dog | 同类动物 | 0.0 | 0.9430 |
| book-fish | 无关 | 0.0 | -0.3162 |

- **One-hot 点积 = 0**：所有不同词完全正交，模型无法区分
- **Dense cos ≈ 1**：近义词方向接近，成功编码了语义相似性
- **负值**：方向几乎相反，比 0 更「不相似」

## Step 6: 可视化对比

左图：one-hot 点积（全 0）；右图：dense 余弦相似度（近义接近 1）

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Colab 兼容：非交互式后端
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

labels = [f'{w1}\n{w2}' for w1, w2, _ in pairs]
oh_values = [dot_product(one_hot[w1], one_hot[w2]) for w1, w2, _ in pairs]
dense_values = [cosine_similarity(dense[w1], dense[w2]) for w1, w2, _ in pairs]

bars1 = ax1.bar(labels, oh_values, color='steelblue', width=0.5)
ax1.set_title('One-hot Dot Products\n(all zero — no similarity encoded)', fontsize=12)
ax1.set_ylabel('Dot Product')
ax1.set_ylim(-0.2, 1.0)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars1, oh_values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

colors = ['#e74c3c' if v > 0.5 else '#f39c12' if v > 0 else '#3498db' for v in dense_values]
bars2 = ax2.bar(labels, dense_values, color=colors, width=0.5)
ax2.set_title('Dense Vector Cosine Similarity\n(similar words → high similarity)', fontsize=12)
ax2.set_ylabel('Cosine Similarity')
ax2.set_ylim(-0.5, 1.2)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars2, dense_values):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('📊 左图全 0 = one-hot 无法编码相似度；右图近义接近 1 = dense 成功编码')

### 读图指南

- **左图**（One-hot）：三组词对的点积全是 0（蓝色柱），无论它们是近义词还是无关词。
- **右图**（Dense）：近义词 hotel-motel 余弦相似度接近 1.0（红色柱最高），同类 cat-dog 也接近 1.0（橙色柱），无关词 book-fish 为负值（蓝色柱）。
- **关键结论**：dense vector 成功把「语义相近 = 方向接近」编码进了向量空间，而 one-hot 完全做不到。

## 和本讲哪个 waypoint 对应

**WP01 — 为什么需要词向量？**

这个 capsule 对应从 one-hot → dense 的动机链：
1. Slides p19: one-hot 向量展示
2. Slides p20: 正交问题 — "no natural notion of similarity"
3. Slides p22: dense vector 解决方案
4. Notes §2.2 Eq.1-2: 数学论证

下一个 waypoint (WP02) 会讲如何从语料中自动学习这样的 dense 向量。

## 容易误解的地方

1. **Dense 向量的值不是手动设计的** — 实际中由模型（如 Word2Vec）从语料中学习得到。这里的 toy 向量只是为了演示「训练后应该有的效果」。

2. **2 维只是演示** — 实际词向量通常 100-300 维。维度越高，能编码的语义信息越丰富。

3. **余弦相似度 vs 点积** — 点积受向量长度影响；余弦相似度归一化了长度，只看方向。one-hot 向量长度都是 1，所以两者等价。

4. **One-hot 不是完全没用** — 在分类任务的输出层，one-hot 仍然用作标签表示。它的问题是作为**输入表示**时无法编码相似度。

5. **book-fish 余弦为负** — 负值表示两个向量方向几乎相反，比 0 更「不相似」。实际训练的词向量中，反义词经常出现负相似度。